# Notebook 2 — Option Data and Descriptive Statistics

**Before this notebook:** finish Notebook 1. We assume you can make a DataFrame,
filter it, and use groupby.

**What we do here**

1. Pull a live NIFTY option chain from the NSE website
2. Clean it into a tidy table
3. Work through every descriptive statistic, with an explanation of what each one
   actually tells you:
   - **Centre** — mean, median, mode, weighted mean
   - **Spread** — range, variance, standard deviation, IQR, MAD, coefficient of variation
   - **Shape** — skewness (left and right), kurtosis
   - **Position** — quantiles, percentiles, z-scores, outliers
   - **Relationships** — covariance, correlation
4. One fully solved question
5. One practice question

**The point of descriptive statistics:** you have hundreds of numbers. Nobody can
look at hundreds of numbers and understand them. Descriptive statistics squeeze
them into a handful of figures that describe the whole set — where it sits, how
spread out it is, and what shape it has.

In [67]:
pip install numpy pandas matplotlib scipy requests openpyxl yfinance


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [68]:
# !pip install numpy pandas matplotlib scipy requests openpyxl yfinance

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy import stats

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Ready.")

Ready.


---
# 1. What option data looks like

An **option chain** is a table of every option contract available on an underlying,
for a given expiry. For NIFTY you get one row per strike, with a call and a put side.

| Column | What it means |
|---|---|
| `strike` | The price at which the option can be exercised |
| `type` | `CE` = Call, `PE` = Put |
| `expiry` | The date the contract dies |
| `ltp` | Last traded price — what the option itself costs |
| `iv` | Implied volatility (%) — how much movement the market is pricing in |
| `oi` | Open interest — contracts currently outstanding |
| `change_oi` | Change in open interest since the previous day |
| `volume` | Contracts traded today |
| `spot` | Where the underlying index actually is right now |

**Spot** matters because it tells you which strikes are near the money. Strikes far
from spot are barely traded, and their numbers are unreliable — a point that comes
back when we start cleaning.

## 1.1 Fetching the chain from NSE

NSE's website has a free endpoint at
`https://www.nseindia.com/api/option-chain-indices?symbol=NIFTY`.

Two things make it awkward:

1. **It rejects anything that does not look like a browser.** We send a
   `User-Agent` header to look like Chrome.
2. **It needs a cookie first.** We visit the homepage with a `Session` object,
   which stores the cookie, and only then request the data.

If it still fails — NSE blocks a lot of cloud IPs, so Google Colab often will not
work — the function below falls back to a realistic made-up chain so the rest of
the notebook still runs. Everything you learn applies either way.

In [71]:
pip install curl_cffi


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [72]:
import requests
import time

# Base headers that mimic a real browser navigating to the site
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
}

def fetch_nse_chain(symbol="NIFTY"):
    try:
        session = requests.Session()
        session.headers.update(HEADERS)
        
        # 1. Hit the homepage first to establish session and grab cookies
        session.get("https://www.nseindia.com", timeout=10)
        
        # 2. Crucial Step: Wait a moment so the WAF doesn't flag us as a bot
        time.sleep(2) 
        
        # 3. Update headers to mimic an internal AJAX API request
        session.headers.update({
            "Accept": "application/json, text/javascript, */*; q=0.01",
            "Sec-Fetch-Dest": "empty",
            "Sec-Fetch-Mode": "cors",
            "Sec-Fetch-Site": "same-origin",
            "X-Requested-With": "XMLHttpRequest"
        })
        
        # 4. Hit the Option Chain API
        url = f"https://www.nseindia.com/api/option-chain-indices?symbol={symbol}"
        response = session.get(url, timeout=10)
        
        if response.status_code == 200:
            return response.json()
            
        print(f"NSE replied with status {response.status_code}")
        return None
        
    except Exception as e:
        print("Could not reach NSE:", e)
        return None


raw = fetch_nse_chain("NIFTY")
print("Got live data!" if raw else "Live data unavailable - we will use the backup below")

NSE replied with status 404
Live data unavailable - we will use the backup below


## 1.2 Turning the JSON into a table

The JSON is nested: `raw["records"]["data"]` is a list, and each item in it holds a
`CE` dictionary and a `PE` dictionary. We loop through and flatten it into rows.

This shape — a loop that builds a list of dictionaries, then one `pd.DataFrame(rows)`
at the end — is the standard way to flatten any API response.

In [74]:
def chain_to_dataframe(raw_json):
    rows = []
    spot = raw_json["records"]["underlyingValue"]

    for item in raw_json["records"]["data"]:
        for side in ["CE", "PE"]:
            if side in item:
                d = item[side]
                rows.append({
                    "type":      side,
                    "strike":    d["strikePrice"],
                    "expiry":    d["expiryDate"],
                    "ltp":       d["lastPrice"],
                    "iv":        d["impliedVolatility"],
                    "oi":        d["openInterest"],
                    "change_oi": d["changeinOpenInterest"],
                    "volume":    d["totalTradedVolume"],
                })

    df = pd.DataFrame(rows)
    df["spot"] = spot
    return df


def make_backup_chain(spot=24500):
    # A made-up but realistic chain, used only if NSE is unreachable.
    rng = np.random.default_rng(7)
    strikes = np.arange(spot - 1500, spot + 1550, 50)
    rows = []

    for k in strikes:
        moneyness = (k - spot) / spot
        smile = 12 + 45 * moneyness ** 2                 # IV is lowest near the moneyx
        distance = abs(k - spot)

        for side in ["CE", "PE"]:
            intrinsic = max(spot - k, 0) if side == "CE" else max(k - spot, 0)
            time_value = 180 * np.exp(-((k - spot) / 700) ** 2)
            rows.append({
                "type": side,
                "strike": int(k),
                "expiry": "30-Jan-2025",
                "ltp": round(max(intrinsic + time_value + rng.normal(0, 4), 0.05), 2),
                "iv": round(max(smile + rng.normal(0, 1.2), 0), 2),
                "oi": int(max(rng.normal(300000 * np.exp(-(distance / 600) ** 2), 40000), 0)),
                "change_oi": int(rng.normal(0, 45000)),
                "volume": int(max(rng.normal(90000 * np.exp(-(distance / 500) ** 2), 15000), 0)),
            })

    df = pd.DataFrame(rows)
    df["spot"] = spot
    return df


if raw:
    chain = chain_to_dataframe(raw)
    source = "live NSE"
else:
    chain = make_backup_chain()
    source = "backup sample"

print(f"Source: {source}")
print("Shape:", chain.shape)
chain.head()

Source: backup sample
Shape: (122, 9)


,type,strike,expiry,ltp,iv,oi,change_oi,volume,spot
0,CE,23000,30-Jan-2025,"1,501.83",12.53,0,-40076,0,24500
1,PE,23000,30-Jan-2025,0.05,12.24,54187,-22149,0,24500
2,CE,23050,30-Jan-2025,"1,454.42",12.59,5088,-41871,0,24500
3,PE,23050,30-Jan-2025,5.25,10.54,0,-85555,0,24500
4,CE,23100,30-Jan-2025,"1,395.93",11.86,0,12206,2386,24500


## 1.3 Cleaning

Three things to fix, and each one has a reason:

1. **One expiry at a time.** The chain holds several expiries. Mixing them makes any
   statistic meaningless, because a 7-day option and a 90-day option are different animals.
2. **Drop `iv == 0`.** NSE reports zero implied volatility for strikes with no trades.
   That is not a real zero — it is a missing value wearing a disguise. Leave them in
   and your average IV gets dragged towards nothing.
3. **Keep strikes near spot.** Deep out-of-the-money strikes have stale prices.

Point 2 is the important one. Fake zeros are more dangerous than obvious `NaN`s
precisely because nothing warns you about them.

In [76]:
print("Expiries available:", chain["expiry"].unique()[:5])

nearest = chain["expiry"].iloc[0]
df = chain[chain["expiry"] == nearest].copy()
print(f"\nUsing expiry: {nearest}  ->  {len(df)} rows")

spot = df["spot"].iloc[0]
print(f"Spot: {spot:,.2f}")

# Fake zeros
print("\nRows with iv = 0 :", (df["iv"] == 0).sum())
df = df[df["iv"] > 0].copy()
print("Rows left        :", len(df))

# Keep strikes within 1000 points of spot
df = df[(df["strike"] > spot - 1000) & (df["strike"] < spot + 1000)].copy()
df["moneyness"] = (df["strike"] - spot).round(0)
df = df.sort_values(["type", "strike"]).reset_index(drop=True)

print("Final shape:", df.shape)
df.head()

Expiries available: ['30-Jan-2025']

Using expiry: 30-Jan-2025  ->  122 rows
Spot: 24,500.00

Rows with iv = 0 : 0
Rows left        : 122
Final shape: (78, 10)


,type,strike,expiry,ltp,iv,oi,change_oi,volume,spot,moneyness
0,CE,23550,30-Jan-2025,981.34,12.69,0,-3563,2963,24500,-950
1,CE,23600,30-Jan-2025,934.82,11.35,26875,-89898,0,24500,-900
2,CE,23650,30-Jan-2025,887.82,12.99,45557,-69157,23739,24500,-850
3,CE,23700,30-Jan-2025,853.15,11.40,48656,-35698,0,24500,-800
4,CE,23750,30-Jan-2025,804.33,11.65,40474,358,3856,24500,-750


In [77]:
# Split into calls and puts - we use these throughout
calls = df[df["type"] == "CE"].copy()
puts = df[df["type"] == "PE"].copy()

print(f"Calls: {len(calls)}   Puts: {len(puts)}")

# The main column we analyse
call_iv = calls["iv"].values
print("\nCall IVs:", np.round(call_iv[:10], 2), "...")

Calls: 39   Puts: 39

Call IVs: [12.69 11.35 12.99 11.4  11.65 12.44 12.58 11.42 13.16 13.05] ...


---
# 2. Measures of Centre

The first question about any set of numbers: **where is the middle?**

There are three answers, and they disagree with each other in a way that is
informative rather than annoying.

## 2.1 Mean

The **mean** is the average: add everything up, divide by how many there are.

$$ \text{mean} = \frac{x_1 + x_2 + \dots + x_n}{n} $$

**What it is good for:** it uses every value, so nothing is ignored.

**Its weakness:** it is dragged around by extreme values. One enormous number pulls
the mean up even if every other value is small. Nine people earning Rs 50,000 and
one earning Rs 50 lakh have a mean salary of Rs 5.45 lakh — a figure that describes
nobody in the room.

In [80]:
values = calls["iv"]

print("Using pandas :", values.mean())
print("Using numpy  :", np.mean(values))
print("By hand      :", values.sum() / len(values))

# Watch what one extreme value does
demo = pd.Series([12, 13, 14, 13, 12, 15, 14])
print("\nOriginal mean         :", demo.mean().round(2))
demo_with_outlier = pd.Series([12, 13, 14, 13, 12, 15, 14, 95])
print("After adding one 95   :", demo_with_outlier.mean().round(2))

Using pandas : 11.942051282051283
Using numpy  : 11.942051282051283
By hand      : 11.942051282051283

Original mean         : 13.29
After adding one 95   : 23.5


## 2.2 Median

The **median** is the middle value once you sort everything. Half the data is
below it, half above.

If there is an even number of values, it is the average of the middle two.

**Its strength is exactly the mean's weakness:** it does not care how extreme the
extremes are, only how many values sit on each side. This makes it the honest
choice for skewed data — which is why house prices and salaries are always quoted
as medians.

In [82]:
print("Median IV:", values.median())

print("\nSame demo as before:")
print("Mean with the outlier   :", round(demo_with_outlier.mean(), 2))
print("Median with the outlier :", round(demo_with_outlier.median(), 2))
print("Median without it       :", round(demo.median(), 2))
print("\nThe mean moved a lot. The median barely noticed.")

Median IV: 11.76

Same demo as before:
Mean with the outlier   : 23.5
Median with the outlier : 13.5
Median without it       : 13.0

The mean moved a lot. The median barely noticed.


### Comparing the mean and the median

This comparison is a free diagnostic, and worth doing every time:

| What you see | What it means |
|---|---|
| mean ≈ median | The data is roughly symmetric |
| mean **>** median | Right-skewed — a tail of large values pulling the mean up |
| mean **<** median | Left-skewed — a tail of small values pulling the mean down |

We come back to this in the skewness section, where it gets a number attached.

In [84]:
for name, series in [("Call IV", calls["iv"]), ("Put IV", puts["iv"]),
                     ("Call OI", calls["oi"]), ("Call LTP", calls["ltp"])]:
    m, md_ = series.mean(), series.median()
    if m > md_ * 1.02:
        verdict = "right-skewed"
    elif m < md_ * 0.98:
        verdict = "left-skewed"
    else:
        verdict = "roughly symmetric"
    print(f"{name:10} mean={m:12,.2f}  median={md_:12,.2f}  ->  {verdict}")

Call IV    mean=       11.94  median=       11.76  ->  roughly symmetric
Put IV     mean=       11.98  median=       12.05  ->  roughly symmetric
Call OI    mean=  163,967.15  median=  135,567.00  ->  right-skewed
Call LTP   mean=      351.65  median=      186.05  ->  right-skewed


## 2.3 Mode

The **mode** is the value that occurs most often.

For continuous data like IV it is often useless — every value is slightly different,
so nothing repeats. It earns its keep with **categories** (which option type is most
common) and with values that repeat by design (strike prices, round-number prices).

`.mode()` returns a Series, because data can have two or more equally common values.
That is why you see `.mode()[0]` — taking the first one.

In [86]:
print("Most common option type:", df["type"].mode()[0])

# On continuous data, bucket it first - then the mode becomes meaningful
iv_buckets = pd.cut(calls["iv"], bins=8)
print("\nMost common IV range:", iv_buckets.mode()[0])
print("\nHow the IVs fall into buckets:")
print(iv_buckets.value_counts().sort_index())

# scipy gives you the count as well
result = stats.mode(calls["ltp"].round(0), keepdims=True)
print(f"\nMost common rounded LTP: {result.mode[0]}, appearing {result.count[0]} times")

Most common option type: CE

Most common IV range: (12.641, 13.178]

How the IVs fall into buckets:
iv
(9.956, 10.496]      3
(10.496, 11.032]     6
(11.032, 11.569]     8
(11.569, 12.105]     4
(12.105, 12.641]     3
(12.641, 13.178]    13
(13.178, 13.714]     0
(13.714, 14.25]      2
Name: count, dtype: int64

Most common rounded LTP: 32.0, appearing 1 times


## 2.4 Weighted mean

An ordinary mean treats every value as equally important. A **weighted mean** lets
some values count more than others:

$$ \text{weighted mean} = \frac{\sum (x_i \times w_i)}{\sum w_i} $$

In options this is genuinely useful. Averaging IV across all strikes treats a strike
with 5 contracts outstanding the same as one with 500,000. Weighting by open interest
gives you the IV the market's money is actually sitting at.

In [88]:
simple_iv = calls["iv"].mean()
weighted_iv = np.average(calls["iv"], weights=calls["oi"])

print(f"Simple average IV   : {simple_iv:.2f}%")
print(f"OI-weighted average : {weighted_iv:.2f}%")
print(f"Difference          : {weighted_iv - simple_iv:+.2f} percentage points")

# The OI-weighted average strike - roughly where positioning is concentrated
weighted_strike = np.average(df["strike"], weights=df["oi"])
print(f"\nSpot                    : {spot:,.0f}")
print(f"OI-weighted mean strike : {weighted_strike:,.0f}")

Simple average IV   : 11.94%
OI-weighted average : 12.09%
Difference          : +0.15 percentage points

Spot                    : 24,500
OI-weighted mean strike : 24,490


---
# 3. Measures of Spread

Knowing the centre is not enough. These two sets both average 50:

- `49, 50, 51`
- `10, 50, 90`

They are completely different data. **Spread** is what separates them.

## 3.1 Range

**Range = maximum − minimum.** The simplest possible measure.

Instantly understandable, and almost useless on its own: it depends entirely on two
values and ignores everything in between. One freak observation and the range is a
lie about the rest of the data.

In [91]:
print("Max IV  :", calls["iv"].max())
print("Min IV  :", calls["iv"].min())
print("Range   :", (calls["iv"].max() - calls["iv"].min())

print("\nRange of every numeric column:")
print((df[["ltp", "iv", "oi", "volume"]].max() - df[["ltp", "iv", "oi", "volume"]].min()))

Max IV  : 14.25
Min IV  : 9.96


AttributeError: 'float' object has no attribute 'round'

## 3.2 Variance and standard deviation

These are the serious measures of spread.

**Variance** is the average squared distance from the mean:

$$ s^2 = \frac{\sum (x_i - \bar{x})^2}{n - 1} $$

Why square the distances? Because otherwise the positives and negatives cancel out
and you always get zero. Squaring also means big deviations count much more heavily
than small ones.

**Standard deviation** is the square root of the variance. Squaring changed the
units (rupees became rupees-squared); the square root changes them back. That is the
whole reason it exists — it is the readable version of the variance.

### The `ddof` trap

That `n - 1` in the denominator instead of `n` matters:

- `ddof=0` divides by `n` — use when your data is the **entire population**
- `ddof=1` divides by `n-1` — use when your data is a **sample** of a larger population

Nearly all real data is a sample, so `ddof=1` is usually right.

**And here is the trap:** `numpy` defaults to `ddof=0`, `pandas` defaults to `ddof=1`.
The same data gives you two different answers depending on which library you used.
On large datasets the gap is tiny; on 40 rows it is visible. Always pass `ddof`
explicitly and you never have to think about it again.

In [ ]:
values = calls["iv"]

print("--- Variance ---")
print(f"pandas default (ddof=1): {values.var():.4f}")
print(f"numpy default  (ddof=0): {np.var(values):.4f}")
print(f"numpy with ddof=1      : {np.var(values, ddof=1):.4f}   <- matches pandas")

print("\n--- Standard deviation ---")
print(f"pandas (sample) : {values.std():.4f}")
print(f"numpy  (sample) : {np.std(values, ddof=1):.4f}")

print("\n--- By hand, to see what it is doing ---")
deviations = values - values.mean()
squared = deviations ** 2
manual_var = squared.sum() / (len(values) - 1)
print(f"Variance : {manual_var:.4f}")
print(f"Std dev  : {np.sqrt(manual_var):.4f}")

### Reading a standard deviation

For data shaped roughly like a bell curve, the **empirical rule** applies:

- About **68%** of values fall within 1 standard deviation of the mean
- About **95%** within 2
- About **99.7%** within 3

So "mean 14.2%, std dev 2.1%" means most IVs sit between 12.1% and 16.3%, and
anything outside 10.0%–18.4% is unusual.

The rule assumes a bell curve. Our data may not be one — checking that is exactly
what the skewness and kurtosis sections are for.

In [ ]:
mean_iv = values.mean()
std_iv = values.std()

for k in [1, 2, 3]:
    low, high = mean_iv - k * std_iv, mean_iv + k * std_iv
    inside = ((values >= low) & (values <= high)).mean() * 100
    expected = [68.3, 95.4, 99.7][k - 1]
    print(f"Within {k} std ({low:6.2f} to {high:6.2f}): "
          f"{inside:5.1f}% of the data   (bell curve says {expected}%)")

## 3.3 IQR — the interquartile range

**Quartiles** cut sorted data into four equal parts:

- **Q1** (25th percentile) — a quarter of the data is below this
- **Q2** (50th percentile) — the median
- **Q3** (75th percentile) — three quarters below

**IQR = Q3 − Q1**: the width of the middle half of the data.

Because it throws away the top and bottom 25%, extreme values cannot touch it. It is
to the standard deviation what the median is to the mean — the version that survives
outliers.

In [ ]:
q1 = values.quantile(0.25)
q2 = values.quantile(0.50)
q3 = values.quantile(0.75)
iqr = q3 - q1

print(f"Q1  (25%) : {q1:.2f}")
print(f"Q2  (50%) : {q2:.2f}   <- the median")
print(f"Q3  (75%) : {q3:.2f}")
print(f"IQR       : {iqr:.2f}   <- the middle 50% spans this much")

print("\nFive-number summary:")
print(f"  Min: {values.min():.2f} | Q1: {q1:.2f} | Median: {q2:.2f} | "
      f"Q3: {q3:.2f} | Max: {values.max():.2f}")

## 3.4 Two more: MAD and the coefficient of variation

**Mean Absolute Deviation (MAD)** — the average distance from the mean, using
absolute values instead of squares. Easier to explain than standard deviation
("the typical value is this far from average") and less sensitive to extremes,
since it does not square anything.

**Coefficient of Variation (CV)** — standard deviation divided by the mean,
as a percentage:

$$ CV = \frac{s}{\bar{x}} \times 100 $$

This one solves a specific problem: you cannot compare the spread of IV (around 14)
with the spread of open interest (around 300,000) — the units are different, so the
standard deviations are not comparable. CV strips the units out and gives you
**relative** variability, so the comparison becomes fair.

In [ ]:
mad = (values - values.mean()).abs().mean()
cv = values.std() / values.mean() * 100

print(f"MAD : {mad:.4f}")
print(f"Std : {values.std():.4f}   (always larger, because squaring inflates big gaps)")
print(f"CV  : {cv:.2f}%")

print("\nCV lets us compare columns with different units:")
for col in ["ltp", "iv", "oi", "volume"]:
    series = calls[col]
    print(f"  {col:8} mean={series.mean():12,.2f}  "
          f"std={series.std():12,.2f}  CV={series.std()/series.mean()*100:7.2f}%")

---
# 4. Measures of Shape

Centre and spread still do not pin down a dataset. Two sets can share a mean and a
standard deviation and look nothing alike — one symmetric, one lopsided.

**Shape** covers the rest: is it lopsided (skewness), and are its tails fat
(kurtosis)?

## 4.1 Skewness

**Skewness measures lopsidedness.** A perfectly symmetric distribution has a
skewness of 0.

### Right skew (positive skew)

The tail stretches out to the **right**. Most values are small, a few are very large.

- `mean > median > mode`
- Examples: incomes, house prices, option open interest, trading volume

### Left skew (negative skew)

The tail stretches to the **left**. Most values are large, a few are very small.

- `mean < median < mode`
- Examples: age at death, exam scores on an easy paper, **daily stock returns**

### Which way does the tail point?

The name follows the **tail**, not the bulk of the data. Right skew means the tail
is on the right, even though most of the data is bunched on the left. People get
this backwards constantly.

A memory aid: the skew points where the mean has been dragged.

### How big is big?

| Skewness | Interpretation |
|---|---|
| −0.5 to +0.5 | Fairly symmetric |
| ±0.5 to ±1.0 | Moderately skewed |
| beyond ±1.0 | Highly skewed |

In [ ]:
# Three made-up distributions so you can see the difference cleanly
rng = np.random.default_rng(1)

symmetric  = rng.normal(50, 10, 5000)
right_skew = rng.exponential(10, 5000) + 20        # long tail to the right
left_skew  = 100 - rng.exponential(10, 5000)       # long tail to the left

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, name in zip(axes,
                          [left_skew, symmetric, right_skew],
                          ["LEFT skew (negative)", "SYMMETRIC", "RIGHT skew (positive)"]):
    ax.hist(data, bins=50, color="steelblue", edgecolor="white", alpha=0.85)
    ax.axvline(np.mean(data), color="red", linestyle="--", linewidth=2, label="mean")
    ax.axvline(np.median(data), color="green", linestyle="-", linewidth=2, label="median")
    ax.set_title(f"{name}\nskew = {stats.skew(data):.2f}")
    ax.legend()

plt.tight_layout()
plt.show()

print("Left skew  : mean sits BELOW the median (the left tail drags it down)")
print("Right skew : mean sits ABOVE the median (the right tail drags it up)")

In [ ]:
def describe_skew(value):
    if value > 1:
        return "highly right-skewed"
    if value > 0.5:
        return "moderately right-skewed"
    if value > -0.5:
        return "fairly symmetric"
    if value > -1:
        return "moderately left-skewed"
    return "highly left-skewed"


print("Skewness of our option data\n" + "-" * 55)
for col in ["ltp", "iv", "oi", "volume", "change_oi"]:
    s_pandas = calls[col].skew()                    # sample-adjusted
    s_scipy = stats.skew(calls[col])                # population version
    print(f"{col:11} pandas={s_pandas:7.3f}  scipy={s_scipy:7.3f}  -> {describe_skew(s_pandas)}")

print("\n(pandas and scipy differ slightly because of the same sample-vs-population")
print(" correction we saw with variance. Use scipy.stats.skew(x, bias=False) to match pandas.)")

### Skewness on stock returns

Daily returns of an equity index are the textbook example of **negative skew**:
markets tend to grind upwards in small steps and fall in large ones. Crashes are
sharper than rallies, so the left tail is longer.

This is not a quirk — it is one of the most reliably observed facts in finance.

In [ ]:
import yfinance as yf

nifty = yf.download("^NSEI", period="3y", interval="1d", auto_adjust=True, progress=False)
if isinstance(nifty.columns, pd.MultiIndex):
    nifty.columns = nifty.columns.get_level_values(0)

nifty["ret"] = nifty["Close"].pct_change() * 100
returns = nifty["ret"].dropna()

print(f"Trading days : {len(returns)}")
print(f"Mean         : {returns.mean():.4f}%")
print(f"Median       : {returns.median():.4f}%")
print(f"Std dev      : {returns.std():.4f}%")
print(f"Skewness     : {returns.skew():.4f}  -> {describe_skew(returns.skew())}")
print(f"\nWorst day    : {returns.min():.2f}%")
print(f"Best day     : {returns.max():.2f}%")
print("\nIf the worst day is bigger in size than the best day, that is the negative skew.")

## 4.2 Kurtosis

**Kurtosis measures the tails** — how often you get extreme values compared with a
normal distribution.

Almost everyone reports **excess kurtosis**, which subtracts 3 so that a normal
distribution scores 0 rather than 3. Both `scipy` and `pandas` do this by default.

| Excess kurtosis | Name | What it means |
|---|---|---|
| ≈ 0 | **Mesokurtic** | Tails like a normal distribution |
| > 0 | **Leptokurtic** | **Fat tails** — extreme events happen more often than normal predicts |
| < 0 | **Platykurtic** | **Thin tails** — extremes are rarer, data is more evenly spread |

### Why this is the one that costs money

Financial returns are strongly leptokurtic. A normal distribution says a 5-standard-
deviation daily move should happen roughly once every 7,000 years. In real markets
they turn up every few years.

If you size your risk assuming a bell curve, you are systematically underestimating
how often disaster arrives. Kurtosis is the number that tells you by how much.

In [ ]:
rng = np.random.default_rng(3)

normal_data = rng.normal(0, 1, 5000)
fat_tails = rng.standard_t(df=3, size=5000)          # t-distribution: famously fat-tailed
thin_tails = rng.uniform(-2, 2, 5000)                # uniform: no tails at all

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, data, name in zip(axes,
                          [thin_tails, normal_data, fat_tails],
                          ["PLATYKURTIC (thin)", "MESOKURTIC (normal)", "LEPTOKURTIC (fat)"]):
    ax.hist(data, bins=60, color="darkorange", edgecolor="white", alpha=0.85)
    ax.set_title(f"{name}\nexcess kurtosis = {stats.kurtosis(data):.2f}")
    ax.set_xlim(-6, 6)

plt.tight_layout()
plt.show()

In [ ]:
def describe_kurtosis(value):
    if value > 0.5:
        return "leptokurtic - fat tails, extremes more likely"
    if value < -0.5:
        return "platykurtic - thin tails, extremes rarer"
    return "mesokurtic - roughly normal tails"


print("Kurtosis of our data\n" + "-" * 60)
for col in ["ltp", "iv", "oi", "volume"]:
    k = calls[col].kurtosis()
    print(f"{col:9} excess kurtosis = {k:8.3f}  -> {describe_kurtosis(k)}")

k_ret = returns.kurtosis()
print(f"\nNIFTY daily returns: {k_ret:.3f}  -> {describe_kurtosis(k_ret)}")

print("\nBoth conventions, side by side:")
print(f"  Excess kurtosis (normal = 0) : {stats.kurtosis(returns, fisher=True):.4f}")
print(f"  Raw kurtosis    (normal = 3) : {stats.kurtosis(returns, fisher=False):.4f}")

## 4.3 Testing for normality

Rather than eyeballing skew and kurtosis, you can test formally. The
**Jarque-Bera test** combines both into a single test with a null hypothesis of
"this data is normally distributed".

A **p-value below 0.05** means you reject that — the data is not normal.

The **Shapiro-Wilk test** does a similar job and is more powerful on small samples
(under about 5,000 observations).

Read the p-value as: *if the data really were normal, how likely is a result this
extreme?* Small p-value, unlikely, so the assumption probably fails.

In [ ]:
jb_stat, jb_p = stats.jarque_bera(returns)
print("Jarque-Bera on NIFTY daily returns")
print(f"  statistic = {jb_stat:.2f}")
print(f"  p-value   = {jb_p:.6f}")
print("  ->", "NOT normal" if jb_p < 0.05 else "consistent with normal")

sw_stat, sw_p = stats.shapiro(calls["iv"])
print("\nShapiro-Wilk on call IVs")
print(f"  statistic = {sw_stat:.4f}")
print(f"  p-value   = {sw_p:.6f}")
print("  ->", "NOT normal" if sw_p < 0.05 else "consistent with normal")

---
# 5. Position: percentiles, z-scores and outliers

Centre, spread and shape describe the *whole* dataset. Sometimes the question is
about **one specific value**: is this IV unusually high? Is this strike an outlier?

## 5.1 Percentiles and quantiles

The **p-th percentile** is the value below which p% of the data falls. The 90th
percentile of IV is the level that 90% of strikes sit beneath.

*Quantile* and *percentile* are the same idea on different scales — `quantile(0.9)`
is the 90th percentile.

In [ ]:
for p in [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]:
    print(f"{int(p*100):3d}th percentile of IV: {calls['iv'].quantile(p):7.2f}")

print("\nSame thing with numpy:")
print(np.percentile(calls["iv"], [5, 25, 50, 75, 95]).round(2))

# Where does a specific value sit?
target = calls["iv"].iloc[len(calls) // 2]
pct_rank = (calls["iv"] < target).mean() * 100
print(f"\nAn IV of {target:.2f} sits at the {pct_rank:.1f}th percentile.")

## 5.2 Z-scores

A **z-score** says how many standard deviations a value is from the mean:

$$ z = \frac{x - \bar{x}}{s} $$

- `z = 0` — exactly average
- `z = +2` — two standard deviations above average
- `z = −1.5` — one and a half below

Z-scores are unitless, which is the point: an IV z-score and an open-interest
z-score are directly comparable even though the raw numbers are not.

By convention, `|z| > 3` marks an outlier (`|z| > 2` if you want to be stricter).

In [ ]:
calls["iv_z"] = (calls["iv"] - calls["iv"].mean()) / calls["iv"].std()
calls["oi_z"] = (calls["oi"] - calls["oi"].mean()) / calls["oi"].std()

print(calls[["strike", "iv", "iv_z", "oi", "oi_z"]].head(8))

print(f"\nMean of the z-scores : {calls['iv_z'].mean():.6f}  (always 0)")
print(f"Std of the z-scores  : {calls['iv_z'].std():.6f}  (always 1)")

outliers = calls[calls["iv_z"].abs() > 2]
print(f"\nStrikes with |z| > 2 on IV: {len(outliers)}")
if len(outliers):
    print(outliers[["strike", "iv", "iv_z"]])

## 5.3 Finding outliers with the IQR rule

The other standard method, and the one behind the whiskers on a boxplot:

- **Lower fence** = Q1 − 1.5 × IQR
- **Upper fence** = Q3 + 1.5 × IQR

Anything outside the fences is flagged.

**Which method should you use?** The z-score method relies on the mean and standard
deviation, both of which are themselves distorted by the outliers you are hunting.
The IQR method uses quartiles, which are not. On skewed data, prefer IQR.

In [ ]:
def find_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return low, high, series[(series < low) | (series > high)]


for col in ["iv", "oi", "volume"]:
    low, high, found = find_outliers_iqr(calls[col])
    print(f"{col:8} fences: {low:12,.2f} to {high:12,.2f}   outliers found: {len(found)}")

low, high, iv_outliers = find_outliers_iqr(calls["iv"])
if len(iv_outliers):
    print("\nOutlier strikes by IV:")
    print(calls.loc[iv_outliers.index, ["strike", "iv", "oi"]])
else:
    print("\nNo IV outliers by the IQR rule.")

## 5.4 Boxplots — all of this in one picture

A **boxplot** draws the five-number summary:

- The **box** runs from Q1 to Q3, so its width is the IQR
- The **line inside** is the median
- The **whiskers** reach to the fences
- **Dots** beyond them are outliers

An off-centre median inside the box is skewness you can see at a glance.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].boxplot([calls["iv"], puts["iv"]])
axes[0].set_xticklabels(["Calls", "Puts"])
axes[0].set_title("Implied volatility")
axes[0].set_ylabel("IV %")

axes[1].hist(calls["iv"], bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(calls["iv"].mean(), color="red", linestyle="--", label="mean")
axes[1].axvline(calls["iv"].median(), color="green", label="median")
axes[1].set_title(f"Call IV distribution (skew = {calls['iv'].skew():.2f})")
axes[1].legend()

axes[2].hist(returns, bins=60, color="darkorange", edgecolor="white")
axes[2].axvline(returns.mean(), color="red", linestyle="--", label="mean")
axes[2].axvline(returns.median(), color="green", label="median")
axes[2].set_title(f"NIFTY returns (skew={returns.skew():.2f}, kurt={returns.kurtosis():.2f})")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
# 6. Relationships between columns

Everything so far described one column at a time. These two describe how a **pair**
of columns move together.

## 6.1 Covariance and correlation

**Covariance** tells you the direction of the relationship: positive means they
tend to rise together, negative means one rises as the other falls. Its problem is
that the size of the number depends on the units, so it is impossible to interpret
on its own.

**Correlation** fixes that by dividing covariance by both standard deviations. The
result always sits between **−1 and +1**:

| Value | Meaning |
|---|---|
| +1 | Perfect positive relationship |
| +0.7 to +0.9 | Strong positive |
| +0.3 to +0.7 | Moderate positive |
| −0.3 to +0.3 | Weak or none |
| −1 | Perfect negative relationship |

**Correlation is not causation**, and it only detects *straight-line* relationships.
Two variables can be perfectly related in a curved way and still show a correlation
near zero — which is exactly what happens with the volatility smile.

In [ ]:
print("Covariance matrix (hard to read - the units are all mixed up):")
print(calls[["ltp", "iv", "oi", "volume"]].cov())

print("\n\nCorrelation matrix (always between -1 and +1):")
print(calls[["strike", "ltp", "iv", "oi", "volume"]].corr().round(3))

print(f"\nStrike vs LTP for calls: {calls['strike'].corr(calls['ltp']):.3f}")
print("Strongly negative, as it must be: higher strikes make a call cheaper.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(calls["strike"], calls["ltp"], color="steelblue")
axes[0].set_title(f"Strike vs LTP  (corr = {calls['strike'].corr(calls['ltp']):.2f})")
axes[0].set_xlabel("Strike")
axes[0].set_ylabel("LTP")

axes[1].scatter(calls["strike"], calls["iv"], color="crimson")
axes[1].axvline(spot, color="black", linestyle="--", label="spot")
axes[1].set_title("Strike vs IV - the volatility smile")
axes[1].set_xlabel("Strike")
axes[1].legend()

axes[2].scatter(calls["oi"], calls["volume"], color="green")
axes[2].set_title(f"OI vs Volume  (corr = {calls['oi'].corr(calls['volume']):.2f})")
axes[2].set_xlabel("Open interest")
axes[2].set_ylabel("Volume")

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("The middle chart is the point about linear correlation: strike and IV are")
print("clearly related, but the relationship is a U-shape, so correlation misses it.")

## 6.2 describe() and groupby — everything at once

`.describe()` gives you count, mean, std, min, quartiles and max in one call. It
should be the first thing you run on any new dataset.

It leaves out skewness and kurtosis, so we add those manually below.

Combine it with `groupby` and you get the same summary broken down by category.

In [ ]:
print(calls[["ltp", "iv", "oi", "volume"]].describe().round(2))

In [ ]:
def full_stats(series):
    # Everything describe() gives you, plus the shape measures.
    return pd.Series({
        "count":    series.count(),
        "mean":     series.mean(),
        "median":   series.median(),
        "std":      series.std(),
        "variance": series.var(),
        "min":      series.min(),
        "q1":       series.quantile(0.25),
        "q3":       series.quantile(0.75),
        "max":      series.max(),
        "range":    series.max() - series.min(),
        "iqr":      series.quantile(0.75) - series.quantile(0.25),
        "cv_pct":   series.std() / series.mean() * 100,
        "skew":     series.skew(),
        "kurtosis": series.kurtosis(),
    })


summary = pd.DataFrame({col: full_stats(calls[col]) for col in ["ltp", "iv", "oi", "volume"]})
print(summary.round(3))

In [ ]:
print("--- Calls vs Puts, side by side ---")
print(df.groupby("type")[["ltp", "iv", "oi", "volume"]].agg(["mean", "median", "std"]).round(2))

print("\n--- Shape measures by type ---")
shape = df.groupby("type")["iv"].agg([
    ("mean", "mean"),
    ("median", "median"),
    ("skew", lambda x: x.skew()),
    ("kurtosis", lambda x: x.kurtosis()),
]).round(3)
print(shape)

---
# 7. Solved Question

> **The task**
>
> Produce a full descriptive-statistics report on the NIFTY option chain.
>
> 1. Fetch the chain, keep the nearest expiry, clean out the fake zero IVs.
> 2. For call IV, calculate every measure: mean, median, mode, weighted mean, range,
>    variance, standard deviation, IQR, MAD, CV, skewness, kurtosis.
> 3. State whether the distribution is left- or right-skewed and whether it is
>    lepto-, meso- or platykurtic, and say what that implies.
> 4. Compare calls with puts using groupby.
> 5. Find outlier strikes using both the z-score and IQR methods.
> 6. Calculate the OI-weighted average strike and compare it to spot.
> 7. Run the same analysis on 3 years of NIFTY daily returns and compare the two shapes.
> 8. Export everything to Excel.

### Step 1 — data

In [ ]:
raw2 = fetch_nse_chain("NIFTY")
chain2 = chain_to_dataframe(raw2) if raw2 else make_backup_chain()

nearest = chain2["expiry"].iloc[0]
data = chain2[(chain2["expiry"] == nearest) & (chain2["iv"] > 0)].copy()

spot2 = data["spot"].iloc[0]
data = data[(data["strike"] > spot2 - 1000) & (data["strike"] < spot2 + 1000)]

c_opts = data[data["type"] == "CE"].copy()
p_opts = data[data["type"] == "PE"].copy()

print(f"Expiry : {nearest}")
print(f"Spot   : {spot2:,.2f}")
print(f"Calls  : {len(c_opts)}    Puts: {len(p_opts)}")

### Step 2 — every statistic on call IV

In [ ]:
iv = c_opts["iv"]

report = {
    "Count":                len(iv),
    "Mean":                 iv.mean(),
    "Median":               iv.median(),
    "Mode (bucketed)":      float(pd.cut(iv, bins=8).mode()[0].mid),
    "OI-weighted mean":     np.average(iv, weights=c_opts["oi"]),
    "Minimum":              iv.min(),
    "Maximum":              iv.max(),
    "Range":                iv.max() - iv.min(),
    "Variance (sample)":    iv.var(),
    "Std deviation":        iv.std(),
    "Q1":                   iv.quantile(0.25),
    "Q3":                   iv.quantile(0.75),
    "IQR":                  iv.quantile(0.75) - iv.quantile(0.25),
    "MAD":                  (iv - iv.mean()).abs().mean(),
    "Coeff of variation %": iv.std() / iv.mean() * 100,
    "Skewness":             iv.skew(),
    "Excess kurtosis":      iv.kurtosis(),
}

report_df = pd.DataFrame(report.items(), columns=["Statistic", "Value"])
report_df["Value"] = report_df["Value"].round(4)
print(report_df.to_string(index=False))

### Step 3 — interpretation

The code below turns the numbers into sentences. This step is the one people skip,
and it is the one that matters — a table of statistics nobody has interpreted is
not analysis.

In [ ]:
skew_val = iv.skew()
kurt_val = iv.kurtosis()

print("INTERPRETATION")
print("=" * 60)

print(f"\n1. Centre: mean {iv.mean():.2f}%, median {iv.median():.2f}%")
if iv.mean() > iv.median():
    print("   Mean above median -> pulled up by some high-IV strikes.")
else:
    print("   Mean below median -> pulled down by some low-IV strikes.")

print(f"\n2. Spread: std dev {iv.std():.2f}, CV {iv.std()/iv.mean()*100:.1f}%")
print(f"   Most strikes sit between {iv.mean()-iv.std():.2f}% and {iv.mean()+iv.std():.2f}%.")

print(f"\n3. Skewness: {skew_val:.3f} -> {describe_skew(skew_val)}")
if skew_val > 0.5:
    print("   The right tail is longer: a handful of strikes carry much higher IV")
    print("   than the rest, which is the volatility smile showing up in the statistics.")
elif skew_val < -0.5:
    print("   The left tail is longer: a few strikes have unusually low IV.")
else:
    print("   IV is distributed fairly evenly across these strikes.")

print(f"\n4. Kurtosis: {kurt_val:.3f} -> {describe_kurtosis(kurt_val)}")
if kurt_val > 0.5:
    print("   Fat tails: extreme IV readings are more common than a bell curve predicts,")
    print("   so do not size risk off the normal distribution here.")
elif kurt_val < -0.5:
    print("   Thin tails: IV is spread out fairly uniformly, few extremes.")

### Step 4 — calls vs puts

In [ ]:
comparison = data.groupby("type").agg(
    count=("iv", "count"),
    mean_iv=("iv", "mean"),
    median_iv=("iv", "median"),
    std_iv=("iv", "std"),
    skew_iv=("iv", lambda x: x.skew()),
    kurt_iv=("iv", lambda x: x.kurtosis()),
    total_oi=("oi", "sum"),
    mean_ltp=("ltp", "mean"),
).round(3)

print(comparison)

pcr = p_opts["oi"].sum() / c_opts["oi"].sum()
print(f"\nPut-Call Ratio by OI: {pcr:.3f}")
print("Above 1 means more puts outstanding than calls.")

### Step 5 — outliers, both methods

In [ ]:
c_opts["iv_z"] = (c_opts["iv"] - c_opts["iv"].mean()) / c_opts["iv"].std()

z_outliers = c_opts[c_opts["iv_z"].abs() > 2]
low, high, iqr_out = find_outliers_iqr(c_opts["iv"])
iqr_outliers = c_opts.loc[iqr_out.index]

print(f"Z-score method (|z| > 2) : {len(z_outliers)} strikes")
print(f"IQR method               : {len(iqr_outliers)} strikes")
print(f"IQR fences               : {low:.2f} to {high:.2f}")

both = set(z_outliers.index) & set(iqr_outliers.index)
print(f"Flagged by both methods  : {len(both)}")

if len(z_outliers):
    print("\nZ-score outliers:")
    print(z_outliers[["strike", "iv", "iv_z", "oi"]].round(3).to_string(index=False))

### Step 6 — where is the open interest sitting?

In [ ]:
weighted_strike = np.average(data["strike"], weights=data["oi"])

max_call_oi = c_opts.loc[c_opts["oi"].idxmax()]
max_put_oi = p_opts.loc[p_opts["oi"].idxmax()]

print(f"Spot                     : {spot2:,.0f}")
print(f"OI-weighted mean strike  : {weighted_strike:,.0f}")
print(f"Gap                      : {weighted_strike - spot2:+,.0f} points")
print(f"\nHighest call OI at strike {max_call_oi['strike']:,.0f}  (OI {max_call_oi['oi']:,.0f})")
print(f"Highest put OI at strike  {max_put_oi['strike']:,.0f}  (OI {max_put_oi['oi']:,.0f})")
print("\nThe biggest call OI strike often acts as resistance, the biggest put OI")
print("strike as support - a rule of thumb, not a law.")

### Step 7 — the same treatment on NIFTY returns

In [ ]:
ret_stats = pd.Series({
    "Count":            len(returns),
    "Mean %":           returns.mean(),
    "Median %":         returns.median(),
    "Std dev %":        returns.std(),
    "Annualised vol %": returns.std() * np.sqrt(252),
    "Min %":            returns.min(),
    "Max %":            returns.max(),
    "IQR":              returns.quantile(0.75) - returns.quantile(0.25),
    "Skewness":         returns.skew(),
    "Excess kurtosis":  returns.kurtosis(),
    "% positive days":  (returns > 0).mean() * 100,
}).round(4)

print(ret_stats.to_string())

print(f"\nShape verdict: {describe_skew(returns.skew())}, {describe_kurtosis(returns.kurtosis())}")
print("\nCall IV vs NIFTY returns:")
print(f"  IV      -> skew {iv.skew():+.3f}, kurtosis {iv.kurtosis():+.3f}")
print(f"  Returns -> skew {returns.skew():+.3f}, kurtosis {returns.kurtosis():+.3f}")
print("\nThe returns are the more dangerous distribution: negatively skewed AND")
print("fat-tailed, meaning the rare days are both large and disproportionately down.")

### Step 8 — export

In [ ]:
with pd.ExcelWriter("option_stats_report.xlsx", engine="openpyxl") as writer:
    data.to_excel(writer, sheet_name="Option Chain", index=False)
    report_df.to_excel(writer, sheet_name="Call IV Stats", index=False)
    comparison.to_excel(writer, sheet_name="Calls vs Puts")
    ret_stats.to_frame("Value").to_excel(writer, sheet_name="NIFTY Returns")
    if len(z_outliers):
        z_outliers[["strike", "iv", "iv_z", "oi"]].to_excel(
            writer, sheet_name="Outliers", index=False)

print("Saved option_stats_report.xlsx")
print("Sheets:", pd.ExcelFile("option_stats_report.xlsx").sheet_names)

In [ ]:
# Final picture: four views of the same data
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].hist(c_opts["iv"], bins=20, color="steelblue", edgecolor="white")
axes[0, 0].axvline(iv.mean(), color="red", linestyle="--", label="mean")
axes[0, 0].axvline(iv.median(), color="green", label="median")
axes[0, 0].set_title(f"Call IV  (skew {iv.skew():.2f}, kurt {iv.kurtosis():.2f})")
axes[0, 0].legend()

axes[0, 1].plot(c_opts["strike"], c_opts["iv"], "o-", label="Calls")
axes[0, 1].plot(p_opts["strike"], p_opts["iv"], "s-", label="Puts")
axes[0, 1].axvline(spot2, color="black", linestyle="--", label="spot")
axes[0, 1].set_title("Volatility smile")
axes[0, 1].set_xlabel("Strike")
axes[0, 1].legend()

width = 20
axes[1, 0].bar(c_opts["strike"] - width, c_opts["oi"], width=width * 2, label="Call OI")
axes[1, 0].bar(p_opts["strike"] + width, p_opts["oi"], width=width * 2, label="Put OI")
axes[1, 0].axvline(spot2, color="black", linestyle="--")
axes[1, 0].set_title("Open interest by strike")
axes[1, 0].legend()

axes[1, 1].hist(returns, bins=60, color="darkorange", edgecolor="white")
axes[1, 1].axvline(returns.mean(), color="red", linestyle="--", label="mean")
axes[1, 1].axvline(returns.median(), color="green", label="median")
axes[1, 1].set_title(f"NIFTY returns (skew {returns.skew():.2f}, kurt {returns.kurtosis():.2f})")
axes[1, 1].legend()

for ax in axes.flat:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
# 8. Practice Question

> **The task**
>
> Run the same style of analysis on **BANK NIFTY** option data, and compare it with NIFTY.
>
> 1. Fetch the BANKNIFTY chain: `fetch_nse_chain("BANKNIFTY")`. Clean it the same way
>    (nearest expiry, drop `iv == 0`, keep strikes within 2000 points of spot — BANKNIFTY
>    moves in bigger steps than NIFTY, so the window has to be wider).
> 2. Build a `stats_table(series)` function returning **all** of: count, mean, median,
>    mode (bucketed), range, variance, std dev, Q1, Q3, IQR, MAD, CV, skewness, kurtosis.
> 3. Apply it to BANKNIFTY call IV, put IV, call LTP and call OI. Put the results in one
>    DataFrame with the statistics as rows and the columns as columns.
> 4. For each of those four, write out in a sentence whether it is left- or right-skewed
>    and lepto-/meso-/platykurtic — and say why that shape makes sense for that quantity.
> 5. Find outlier strikes by both the z-score and IQR methods. Where the two disagree,
>    explain which you trust and why.
> 6. Compute the Put-Call Ratio by open interest and by volume. Are they different?
> 7. Download 3 years of `^NSEBANK` daily returns. Compare their skewness and kurtosis
>    with NIFTY's (`^NSEI`). Which index has the more dangerous shape?
> 8. Run a Jarque-Bera test on both return series and state whether either is normal.
> 9. Compute the correlation between NIFTY and BANK NIFTY daily returns. Plot a scatter.
> 10. Export the lot to `banknifty_analysis.xlsx` with at least four sheets.
>
> **Bonus:** compute a **rolling** 30-day standard deviation of BANK NIFTY returns
> (`returns.rolling(30).std()`) and plot it. When was volatility highest?

**Hints**

- Reuse `fetch_nse_chain`, `chain_to_dataframe`, `find_outliers_iqr`, `describe_skew`
  and `describe_kurtosis` — they are all defined above and still in memory.
- For step 3: `pd.DataFrame({"call_iv": stats_table(...), "put_iv": stats_table(...)})`
- To align two return series by date: `pd.concat([a, b], axis=1).dropna()`
- Jarque-Bera: `stats.jarque_bera(series)` returns `(statistic, p_value)`.
- If NSE blocks you, `make_backup_chain(spot=52000)` gives you something to work with.

**How to know you got it right**

- BANKNIFTY IV should be higher than NIFTY IV on average — it is the more volatile index.
- Both indices' daily returns should come out negatively skewed with excess kurtosis
  well above 0.
- The Jarque-Bera p-values should be tiny, comfortably rejecting normality.
- NIFTY and BANK NIFTY returns should correlate somewhere around 0.8–0.9.

In [ ]:
# Your answer here.
# Build it in small pieces and print after each one.

# Step 1: fetch and clean BANKNIFTY


# Step 2: the stats_table function
def stats_table(series):
    pass


# Step 3: apply it


# ... keep going

---
## Cheat sheet

| What you want | Code |
|---|---|
| Mean | `s.mean()` |
| Median | `s.median()` |
| Mode | `s.mode()[0]` |
| Weighted mean | `np.average(s, weights=w)` |
| Range | `s.max() - s.min()` |
| Variance (sample) | `s.var()` |
| Std deviation | `s.std()` |
| Std deviation (population) | `s.std(ddof=0)` |
| Q1 / Q3 | `s.quantile(0.25)` / `s.quantile(0.75)` |
| IQR | `s.quantile(.75) - s.quantile(.25)` |
| MAD | `(s - s.mean()).abs().mean()` |
| Coefficient of variation | `s.std() / s.mean() * 100` |
| Skewness | `s.skew()` |
| Kurtosis (excess) | `s.kurtosis()` |
| Z-scores | `(s - s.mean()) / s.std()` |
| Percentile | `s.quantile(0.9)` |
| Everything at once | `df.describe()` |
| Correlation | `df.corr()` |
| Covariance | `df.cov()` |
| Normality test | `stats.jarque_bera(s)` |
| Rolling std dev | `s.rolling(30).std()` |

### The three things worth remembering

1. **Mean vs median tells you the skew** before you calculate anything.
2. **`ddof` differs between numpy and pandas.** Pass it explicitly.
3. **Financial returns are negatively skewed and fat-tailed.** Any model that assumes
   a bell curve is understating how often things go badly wrong.